# DB MoveOptimizer - University GPT API Connection

This notebook establishes a foundation for connecting to the University of Cologne GPT API using LangChain and LangGraph. We'll build a simple workflow that can be extended with more complex logic later.

## Notebook Outline
1. Environment Setup and API Configuration
2. Initialize OpenAI Client with University Endpoint
3. Create Basic Chat Function
4. Set Up LangGraph State and Nodes
5. Build LangGraph Workflow
6. Visualize the Graph in Browser
7. Test the API Connection

## 1. Environment Setup and API Configuration

First, let's install required dependencies and set up our environment variables.

In [1]:
# Install required packages
import subprocess
import sys

def install_packages(packages):
    """Install packages if not already installed."""
    for package in packages:
        try:
            __import__(package.split("==")[0].replace("-", "_"))
            print(f"✓ {package} already installed")
        except ImportError:
            print(f"Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
            print(f"✓ {package} installed")

# Required packages for this notebook
required_packages = [
    "openai",
    "langgraph",
    "langchain",
    "langchain-core",
    "langchain-openai",
]

install_packages(required_packages)

Installing openai...
✓ openai installed
Installing langgraph...
✓ langgraph installed
Installing langchain...
✓ langchain installed
✓ langchain-core already installed
Installing langchain-openai...
✓ langchain-openai installed


In [2]:
import os
from typing import Optional

# University of Cologne GPT Endpoint Configuration
UNI_GPT_BASE_URL = "https://chat.kiconnect.nrw/api/v1"
UNI_GPT_MODEL = "Openai GPT OSS 120B"

# Get API key from environment variable or prompt user
UNI_GPT_API_KEY = os.getenv("UNI_GPT_API_KEY", "")

if not UNI_GPT_API_KEY:
    # For development: prompt user to enter API key
    from getpass import getpass
    UNI_GPT_API_KEY = getpass("Enter your University GPT API Key: ")

# Configuration summary
print("=" * 60)
print("University GPT API Configuration")
print("=" * 60)
print(f"Base URL: {UNI_GPT_BASE_URL}")
print(f"Model: {UNI_GPT_MODEL}")
print(f"API Key: {'***' + UNI_GPT_API_KEY[-8:] if UNI_GPT_API_KEY else 'NOT SET'}")
print("=" * 60)

University GPT API Configuration
Base URL: https://chat.kiconnect.nrw/api/v1
Model: Openai GPT OSS 120B
API Key: ***jQjSDag=


## 2. Initialize OpenAI Client with University Endpoint

Initialize the OpenAI client pointing to the university's custom endpoint instead of the default OpenAI API.

In [3]:
from openai import OpenAI

# Initialize OpenAI client with university endpoint
client = OpenAI(
    base_url=UNI_GPT_BASE_URL,
    api_key=UNI_GPT_API_KEY
)

print("✓ OpenAI client initialized with University endpoint")

✓ OpenAI client initialized with University endpoint


## 3. Create Basic Chat Function

Define a simple chat function that will be used by our LangGraph workflow.

In [4]:
def call_university_gpt(messages, max_new_tokens=512, temperature=0.0):
    """
    Call the University of Cologne GPT endpoint.
    
    Args:
        messages: List of message dicts with 'role' and 'content' keys
        max_new_tokens: Maximum tokens to generate
        temperature: Temperature for sampling (0.0 = deterministic)
    
    Returns:
        str: The generated response
    """
    response = client.chat.completions.create(
        model=UNI_GPT_MODEL,
        messages=messages,
        max_tokens=max_new_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content.strip()

# Test the basic connection
print("Testing basic API connection...")
try:
    test_response = call_university_gpt(
        [{"role": "user", "content": "Say 'Hello from University GPT' and nothing else."}],
        max_new_tokens=50,
        temperature=0.0
    )
    print(f"✓ API Connection successful!")
    print(f"Response: {test_response}")
except Exception as e:
    print(f"✗ API Connection failed: {e}")

Testing basic API connection...
✓ API Connection successful!
Response: Hello from University GPT


## 4. Set Up LangGraph State and Nodes

Define the state schema and create individual node functions for our workflow.

In [5]:
from typing import TypedDict, List, Any
from langgraph.graph import StateGraph, START, END

# Define the state schema for our workflow
class ChatState(TypedDict):
    """State structure for the chat workflow."""
    messages: List[dict]  # List of message dicts with 'role' and 'content'
    response: str  # The final response from the API
    status: str  # Status of the workflow (processing, complete, error)

print("✓ State schema defined: ChatState")

# Define node functions
def process_user_input(state: ChatState) -> ChatState:
    """
    Process user input and prepare for API call.
    This is where you can add preprocessing logic later.
    """
    print(f"Processing {len(state['messages'])} messages...")
    state["status"] = "processing"
    return state

def call_api(state: ChatState) -> ChatState:
    """
    Call the University GPT API with the current messages.
    """
    try:
        response = call_university_gpt(
            messages=state["messages"],
            max_new_tokens=512,
            temperature=0.0
        )
        state["response"] = response
        state["status"] = "complete"
        print(f"✓ API call successful")
    except Exception as e:
        state["response"] = f"Error: {str(e)}"
        state["status"] = "error"
        print(f"✗ API call failed: {e}")
    return state

def format_output(state: ChatState) -> ChatState:
    """
    Format the final output. This is where you can add postprocessing logic.
    """
    if state["status"] == "complete":
        print(f"Output formatted successfully")
    return state

print("✓ Node functions defined: process_user_input, call_api, format_output")

✓ State schema defined: ChatState
✓ Node functions defined: process_user_input, call_api, format_output


## 5. Build LangGraph Workflow

Construct the graph by connecting nodes with edges to define the workflow logic.

In [7]:
# Create the LangGraph workflow
workflow = StateGraph(ChatState)

# Add nodes
workflow.add_node("input_processor", process_user_input)
workflow.add_node("api_caller", call_api)
workflow.add_node("output_formatter", format_output)

# Define the flow: START -> input_processor -> api_caller -> output_formatter -> END
workflow.add_edge(START, "input_processor")
workflow.add_edge("input_processor", "api_caller")
workflow.add_edge("api_caller", "output_formatter")
workflow.add_edge("output_formatter", END)

# Compile the graph
graph = workflow.compile()

print("✓ LangGraph workflow compiled successfully")
print("\nWorkflow structure:")
print("  START → input_processor → api_caller → output_formatter → END")

✓ LangGraph workflow compiled successfully

Workflow structure:
  START → input_processor → api_caller → output_formatter → END


## 6. Visualize the Graph in Browser

Use LangGraph's built-in visualization to display the workflow structure interactively.

In [8]:
import json

# Get the graph visualization in Mermaid format
def visualize_graph_mermaid(graph):
    """Generate a Mermaid diagram of the graph structure."""
    mermaid = """
graph TD
    Start([START]) --> InputProcessor["input_processor<br/>(Process User Input)"]
    InputProcessor --> APICaller["api_caller<br/>(Call University GPT)"]
    APICaller --> OutputFormatter["output_formatter<br/>(Format Output)"]
    OutputFormatter --> End([END])
    
    style Start fill:#90EE90
    style End fill:#FFB6C6
    style InputProcessor fill:#87CEEB
    style APICaller fill:#FFD700
    style OutputFormatter fill:#DDA0DD
"""
    return mermaid.strip()

# Display the Mermaid diagram
mermaid_diagram = visualize_graph_mermaid(graph)
print("Graph Structure (Mermaid):")
print(mermaid_diagram)
print("\n" + "="*60)

# ASCII visualization
print("\nGraph Structure (ASCII):")
print("""
┌─────────────┐
│   START     │
└──────┬──────┘
       │
       ▼
┌──────────────────────────┐
│  input_processor         │
│  (Process User Input)    │
└──────┬───────────────────┘
       │
       ▼
┌──────────────────────────┐
│  api_caller              │
│  (Call University GPT)   │
└──────┬───────────────────┘
       │
       ▼
┌──────────────────────────┐
│  output_formatter        │
│  (Format Output)         │
└──────┬───────────────────┘
       │
       ▼
┌─────────────┐
│   END       │
└─────────────┘
""")
print("="*60)

# Interactive visualization info
print("\n📊 Interactive Graph Visualization:")
print("-" * 60)
print("To view the graph in your browser, you have two options:")
print("\n1. Use LangGraph Studio (Recommended):")
print("   - This graph is compatible with LangGraph Studio")
print("   - You can view and debug execution flow there")
print("\n2. Copy the Mermaid diagram above:")
print("   - Paste it into mermaid.live")
print("   - Or embed it in markdown files")
print("-" * 60)

Graph Structure (Mermaid):
graph TD
    Start([START]) --> InputProcessor["input_processor<br/>(Process User Input)"]
    InputProcessor --> APICaller["api_caller<br/>(Call University GPT)"]
    APICaller --> OutputFormatter["output_formatter<br/>(Format Output)"]
    OutputFormatter --> End([END])

    style Start fill:#90EE90
    style End fill:#FFB6C6
    style InputProcessor fill:#87CEEB
    style APICaller fill:#FFD700
    style OutputFormatter fill:#DDA0DD


Graph Structure (ASCII):

┌─────────────┐
│   START     │
└──────┬──────┘
       │
       ▼
┌──────────────────────────┐
│  input_processor         │
│  (Process User Input)    │
└──────┬───────────────────┘
       │
       ▼
┌──────────────────────────┐
│  api_caller              │
│  (Call University GPT)   │
└──────┬───────────────────┘
       │
       ▼
┌──────────────────────────┐
│  output_formatter        │
│  (Format Output)         │
└──────┬───────────────────┘
       │
       ▼
┌─────────────┐
│   END       │
└────

## 7. Test the API Connection

Execute test queries through the LangGraph workflow to verify everything is working.

In [9]:
# Test 1: Simple greeting
print("\n" + "="*60)
print("TEST 1: Simple greeting through LangGraph")
print("="*60)

initial_state = {
    "messages": [
        {"role": "user", "content": "What is the capital of France? Answer in one sentence."}
    ],
    "response": "",
    "status": "pending"
}

result = graph.invoke(initial_state)

print(f"\nUser Query: {initial_state['messages'][0]['content']}")
print(f"Status: {result['status']}")
print(f"Response: {result['response']}")

# Test 2: Multi-turn conversation
print("\n" + "="*60)
print("TEST 2: Multi-turn conversation")
print("="*60)

conversation_state = {
    "messages": [
        {"role": "user", "content": "What is machine learning?"},
        {"role": "assistant", "content": "Machine learning is a subset of artificial intelligence..."},
        {"role": "user", "content": "Can you give a real-world example?"}
    ],
    "response": "",
    "status": "pending"
}

result = graph.invoke(conversation_state)

print(f"User Query: {conversation_state['messages'][-1]['content']}")
print(f"Status: {result['status']}")
print(f"Response: {result['response']}")

print("\n" + "="*60)
print("✓ All tests completed!")
print("="*60)


TEST 1: Simple greeting through LangGraph
Processing 1 messages...
✓ API call successful
Output formatted successfully

User Query: What is the capital of France? Answer in one sentence.
Status: complete
Response: The capital of France is Paris.

TEST 2: Multi-turn conversation
Processing 3 messages...
✓ API call successful
Output formatted successfully
User Query: Can you give a real-world example?
Status: complete
Response: ### Real‑World Example: **Movie‑Recommendation Engine (e.g., Netflix, YouTube, Spotify)**  

| Step | What Happens | Why It’s Machine Learning |
|------|--------------|---------------------------|
| **1. Data Collection** | Every time you watch a video, click “like,” skip a song, or finish a movie, the platform records: <br>• User ID <br>• Item ID (movie, song, video) <br>• Interaction type (play, pause, rating, etc.) <br>• Timestamp, device, location, etc. | The system gathers **large, noisy, high‑dimensional data** that no human could manually analyze. |
| **2.

## Next Steps & Future Expansion

This notebook establishes the foundation. Here's how to extend it:

### Phase 2: Enhanced Logic
- **Add conditional routing**: Create decision nodes that route to different paths based on input
- **Implement memory management**: Store conversation history for multi-turn context
- **Add tool calls**: Create agent nodes that can perform specific tasks
- **Implement parallel processing**: Run multiple nodes simultaneously

### Phase 3: DB MoveOptimizer Integration
- **Analyst Agent**: Pattern detection from travel logs
- **Forecaster Agent**: 6-month demand predictions
- **Optimizer Agent**: Portfolio optimization
- **Communicator Agent**: Personalized recommendations

### Visualization Tools
- **LangGraph Studio**: For interactive debugging and visualization
- **mermaid.live**: For quick diagram sharing
- **Custom dashboards**: For monitoring agent execution

### Resources
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [OpenAI API Reference](https://platform.openai.com/docs/api-reference)
- [LangChain Documentation](https://python.langchain.com/docs/)

Happy experimenting! 🚀

## Next Steps & Future Expansion

This notebook establishes the foundation. Here's how to extend it:

### Phase 2: Enhanced Logic
- **Add conditional routing**: Create decision nodes that route to different paths based on input
- **Implement memory management**: Store conversation history for multi-turn context
- **Add tool calls**: Create agent nodes that can perform specific tasks
- **Implement parallel processing**: Run multiple nodes simultaneously

### Phase 3: DB MoveOptimizer Integration
- **Analyst Agent**: Pattern detection from travel logs
- **Forecaster Agent**: 6-month demand predictions
- **Optimizer Agent**: Portfolio optimization
- **Communicator Agent**: Personalized recommendations

### Visualization Tools
- **LangGraph Studio**: For interactive debugging and visualization
- **mermaid.live**: For quick diagram sharing
- **Custom dashboards**: For monitoring agent execution

### Resources
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [OpenAI API Reference](https://platform.openai.com/docs/api-reference)
- [LangChain Documentation](https://python.langchain.com/docs/)

Happy experimenting! 🚀

### Quick Start Guide for `langgraph dev`

The module has been created. Here's what to do:

#### Terminal Commands:
```bash
# 1. Set your API key (already done above)
export UNI_GPT_API_KEY=""

# 2. Navigate to the module directory
cd langgraph_module

# 3. Start the development server
langgraph dev

Then open your browser to **http://localhost:2024**

#### What You'll See:
- **Graph Visualization** - Your workflow drawn interactively
- **Input Panel** - Test your graph with custom inputs
- **Execution Viewer** - Watch state flow through each node
- **Debug Info** - See what happens at each step